In [1]:
import requests
import time

### GET complete data


In [6]:
url = "http://localhost:8000/data"

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "User-Agent": "MyApp/1.0"
}

response = requests.get(url, headers=headers)

In [7]:
print(response.status_code)
print(response.json())

200
{'data': {'user1': {'name': 'Alice', 'age': 25, 'city': 'New York'}, 'user2': {'name': 'Bob', 'age': 30, 'city': 'San Francisco'}, 'user3': {'name': 'Charlie', 'age': 35, 'city': 'London'}, 'product1': {'name': 'Laptop', 'price': 999.99, 'stock': 10}, 'product2': {'name': 'Phone', 'price': 699.99, 'stock': 25}, 'config': {'theme': 'dark', 'language': 'en', 'notifications': True}}, 'count': 6, 'status': 'success'}


### With keep alive

In [8]:
url = "http://localhost:8000/data"

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "User-Agent": "MyApp/1.0",
    "Connection": "keep-alive"
}

# Use Session to reuse TCP connection
session = requests.Session()

for i in range(2):
    start = time.perf_counter()
    response = session.get(url, headers=headers)
    end = time.perf_counter()
    
    print(f"Request {i+1}: {response.json()}, Time taken = {end - start:.4f} seconds")

Request 1: {'data': {'user1': {'name': 'Alice', 'age': 25, 'city': 'New York'}, 'user2': {'name': 'Bob', 'age': 30, 'city': 'San Francisco'}, 'user3': {'name': 'Charlie', 'age': 35, 'city': 'London'}, 'product1': {'name': 'Laptop', 'price': 999.99, 'stock': 10}, 'product2': {'name': 'Phone', 'price': 699.99, 'stock': 25}, 'config': {'theme': 'dark', 'language': 'en', 'notifications': True}}, 'count': 6, 'status': 'success'}, Time taken = 2.0480 seconds
Request 2: {'data': {'user1': {'name': 'Alice', 'age': 25, 'city': 'New York'}, 'user2': {'name': 'Bob', 'age': 30, 'city': 'San Francisco'}, 'user3': {'name': 'Charlie', 'age': 35, 'city': 'London'}, 'product1': {'name': 'Laptop', 'price': 999.99, 'stock': 10}, 'product2': {'name': 'Phone', 'price': 699.99, 'stock': 25}, 'config': {'theme': 'dark', 'language': 'en', 'notifications': True}}, 'count': 6, 'status': 'success'}, Time taken = 0.0031 seconds


### GET a user

In [ ]:
url = "http://localhost:8000/data/user1"

headers = {
    "Accept": "application/json",
    "User-Agent": "MyApp/1.0",
    "Connection": "close"
}

# Run multiple requests
for i in range(5):
    start = time.perf_counter()
    response = requests.get(url, headers=headers)
    elapsed = time.perf_counter() - start

    print(f"Request {i+1}: {response.status_code}, Time taken = {elapsed:.4f} seconds")

Request 1: 200, Time taken = 2.0476 seconds
Request 2: 200, Time taken = 2.0469 seconds
Request 3: 200, Time taken = 2.0379 seconds
Request 4: 200, Time taken = 2.0487 seconds
Request 5: 200, Time taken = 2.0413 seconds


In [12]:
response = requests.get("http://localhost:8000/data/user1")
print(response.status_code)
print(response.json())

200
{'key': 'user1', 'value': {'name': 'Alice', 'age': 25, 'city': 'New York'}, 'status': 'success'}


### POST request

In [ ]:
url = "http://localhost:8000/data/user10"

# Warm-up (ensure key exists, update if needed)
data = {"key": {"name": "Alice Updated", "age": 26}}

requests.post("http://localhost:8000/data/user10", json=data)

# Loop over requests
for i in range(5):
    start = time.perf_counter()
    response = requests.get(url)
    end = time.perf_counter()

    print(
        f"Request {i+1}: {response.status_code}, "
        f"Time = {end - start:.5f}s, "
        f"Cache-Control = {response.headers.get('Cache-Control')}"
    )
    time.sleep(1)  # optional delay to observe effect

Request 1: 404, Time = 2.03015s, Cache-Control = None
Request 2: 404, Time = 2.05267s, Cache-Control = None
Request 3: 404, Time = 2.02862s, Cache-Control = None
Request 4: 404, Time = 2.04366s, Cache-Control = None
Request 5: 404, Time = 2.05549s, Cache-Control = None


In [6]:
print(response.status_code)
print(response.json())

200
{'key': 'user1', 'old_value': {'name': 'Alice', 'age': 25, 'city': 'New York'}, 'new_value': {'name': 'Alice Updated', 'age': 26}, 'status': 'updated'}


### Wrong POST

In [9]:
wrong_data = {"name": "Alice", "age": 25}  # Should be wrapped in "value"

response = requests.post("http://localhost:8000/data/user1", json=wrong_data)

In [10]:
print(f"Status Code: {response.status_code}")
print(f"Response: {response.json()}")

Status Code: 422
Response: {'detail': [{'type': 'missing', 'loc': ['body', 'value'], 'msg': 'Field required', 'input': {'name': 'Alice', 'age': 25}}]}
